# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print("Dataset Metadata Loaded Successfully!\n")

print(f"Name: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets and their fields, including their `@id`s.

We use each entity's `@id` for references, as required by FAIR data principles and the Croissant specification.

In [ ]:
# Display available record sets and their field details:
print("Available record sets (by @id):\n")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"• Record set name: {rs.name}\n  @id: {rs['@id']}")
    field_ids = []
    if hasattr(rs, 'fields') and rs.fields:
        for field in rs.fields:
            field_ids.append(field['@id'])
        print(f"  Fields (@id): {field_ids}\n")
    else:
        print("  No fields found.\n")

# For the rest of the notebook, let's focus on the FIRST record set
if len(record_sets) > 0:
    main_rs = record_sets[0]
    main_rs_id = main_rs['@id']
    main_fields = main_rs.fields
    print(f"\nMain record set selected: {main_rs.name} (@id: {main_rs_id})")
else:
    main_rs = None
    main_rs_id = None
    main_fields = []

## 3. Data Extraction
Load data from the selected record set into a DataFrame. We use the record set and field `@id`s identified above.

In [ ]:
# Extract data for each record set
if main_rs is None:
    print("No record set found. Please check if the dataset provides any records.")
else:
    # List all record set @ids
    record_set_ids = [rs['@id'] for rs in record_sets]
    dataframes = {}
    for rs_id in record_set_ids:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set {rs_id}")

    # Explore columns in our main DataFrame
    main_df = dataframes[main_rs_id]
    print("\nAvailable columns (field @id OR field name):")
    print(main_df.columns.tolist())

    # Preview
    main_df.head()

## 4. Exploratory Data Analysis (EDA)
We can now process numeric fields from the main record set using their `@id`s. As an example, we choose a field related to the PHQ-9 or GAD-7 scores, which are typically numeric.

_Note: Please refer to the actual column names printed above. For demonstration, we use a field such as `'phq9_total'` or `'gad7_total'`, but you should confirm the exact field `@id` for accurate results._

In [ ]:
# Example: EDA on a numeric field -- update `numeric_field_id` as needed based on overview above
numeric_field_id = None
for c in main_df.columns:
    if "phq9" in c.lower() and "total" in c.lower():
        numeric_field_id = c
        break
if not numeric_field_id:
    for c in main_df.columns:
        if "gad7" in c.lower() and "total" in c.lower():
            numeric_field_id = c
            break
if not numeric_field_id:
    # Try fallback to another numeric column
    for c in main_df.columns:
        if main_df[c].dtype in [int, float]:
            numeric_field_id = c
            break

if numeric_field_id is None:
    print("No obvious numeric field found for EDA. Please check your dataset's fields.")
else:
    print(f"Selected numeric field for EDA: {numeric_field_id}")
    threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].dtype in [int, float] else 10
    # Drop NA for safety
    series = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    filtered_df = main_df[series > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}")
    print(filtered_df.head())

    # Normalization
    mean = series.mean()
    std = series.std()
    filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - mean) / std
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optional group field
    # Try grouping by 'sex'/'gender' or 'level_of_education' if present
    possible_groups = ['sex', 'gender', 'level_of_education', 'village']
    group_field = None
    for g in possible_groups:
        if g in main_df.columns:
            group_field = g
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field} (mean):")
        print(grouped_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    else:
        print("No suitable group field found.")

## 5. Visualization
Visualize the distribution of the selected numeric field and any relationships to grouping variables, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is None:
    print("No numeric field to visualize.")
else:
    plt.figure(figsize=(7,4))
    sns.histplot(pd.to_numeric(main_df[numeric_field_id], errors='coerce').dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group if available
    if group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(data=main_df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
You have now loaded, explored, and visualized the Kilifi County Mental Health FAIR² Survey dataset using the `mlcroissant` library. Use the official field and record set `@id`s for unambiguous references. You can further extend the analysis to additional fields and record sets as needed.

Key takeaways:
- The dataset is well-described by its Croissant schema.
- Data can be filtered, transformed, and grouped using standard pandas and Python tools.
- Use `mlcroissant` for robust FAIR data workflows.